<a href="https://colab.research.google.com/github/ankithacs24/ankithaai43/blob/main/ai_1WN24CS043.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def calculate_cost(state):
    """
    Calculates the cost (number of attacking pairs of queens).
    Since columns and rows are unique in a permutation, we only check diagonals.
    """
    conflicts = 0
    n = len(state)
    for i in range(n):
        for j in range(i + 1, n):
            # Check if they attack each other diagonally
            if abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1
    return conflicts

def hill_climbing_4_queens(initial_state):
    current_state = list(initial_state)
    current_cost = calculate_cost(current_state)

    step = 1
    while True:
        print(f"\n--- STEP {step} ---")
        print(f"Current State: {current_state} | Cost: {current_cost}")
        print("Calculating neighbor costs (by swapping column pairs):")

        best_neighbor = None
        best_cost = current_cost
        best_swap_pair = None

        # Generate all unique pairs of columns to swap in lexicographical order:
        # (0,1), (0,2), (0,3), (1,2), (1,3), (2,3)
        n = len(current_state)
        for i in range(n):
            for j in range(i + 1, n):
                # Create neighbor by swapping elements at indices i and j
                neighbor = list(current_state)
                neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
                neighbor_cost = calculate_cost(neighbor)

                print(f"  Swap (x_{i}, x_{j}) -> Neighbor: {neighbor} | Cost: {neighbor_cost}")

                # We want to minimize the cost (steepest descent).
                # If there's a strict improvement:
                if neighbor_cost < best_cost:
                    best_cost = neighbor_cost
                    best_neighbor = neighbor
                    best_swap_pair = (i, j)
                # Tie-breaking: if cost is equal to the current best_cost found so far in this step,
                # the loop naturally preserves the earlier pair due to lexicographical order (i, j),
                # so we don't overwrite if it's just equal to best_cost.

        # If no neighbor strictly improves the cost, hill climbing terminates (local or global optimum)
        if best_neighbor is None:
            print("\nAlgorithm Terminated: No neighboring state has a lower cost.")
            print(f"Final State: {current_state} with Cost: {current_cost}")
            if current_cost == 0:
                print("Goal configuration successfully reached!")
            else:
                print("Stuck in a local optimum.")
            break

        # Move to the best neighbor
        print(f"\nMoving to best neighbor via swap (x_{best_swap_pair[0]}, x_{best_swap_pair[1]})")
        current_state = best_neighbor
        current_cost = best_cost
        step += 1

# Initial state from the problem text: x0=3, x1=1, x2=2, x3=0
initial_state = [3, 1, 2, 0]
hill_climbing_4_queens(initial_state)



--- STEP 1 ---
Current State: [3, 1, 2, 0] | Cost: 2
Calculating neighbor costs (by swapping column pairs):
  Swap (x_0, x_1) -> Neighbor: [1, 3, 2, 0] | Cost: 1
  Swap (x_0, x_2) -> Neighbor: [2, 1, 3, 0] | Cost: 1
  Swap (x_0, x_3) -> Neighbor: [0, 1, 2, 3] | Cost: 6
  Swap (x_1, x_2) -> Neighbor: [3, 2, 1, 0] | Cost: 6
  Swap (x_1, x_3) -> Neighbor: [3, 0, 2, 1] | Cost: 1
  Swap (x_2, x_3) -> Neighbor: [3, 1, 0, 2] | Cost: 1

Moving to best neighbor via swap (x_0, x_1)

--- STEP 2 ---
Current State: [1, 3, 2, 0] | Cost: 1
Calculating neighbor costs (by swapping column pairs):
  Swap (x_0, x_1) -> Neighbor: [3, 1, 2, 0] | Cost: 2
  Swap (x_0, x_2) -> Neighbor: [2, 3, 1, 0] | Cost: 2
  Swap (x_0, x_3) -> Neighbor: [0, 3, 2, 1] | Cost: 4
  Swap (x_1, x_2) -> Neighbor: [1, 2, 3, 0] | Cost: 4
  Swap (x_1, x_3) -> Neighbor: [1, 0, 2, 3] | Cost: 2
  Swap (x_2, x_3) -> Neighbor: [1, 3, 0, 2] | Cost: 0

Moving to best neighbor via swap (x_2, x_3)

--- STEP 3 ---
Current State: [1, 3, 0, 2] 

In [4]:
import random
import math

def calculate_conflicts(state):
    """
    Calculates the number of attacking pairs of queens.
    State is a list of length 8 where index is the column and value is the row.
    """
    conflicts = 0
    n = len(state)
    for i in range(n):
        for j in range(i + 1, n):
            # Check row conflict or diagonal conflict
            if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1
    return conflicts

def get_random_neighbor(state):
    """
    Generates a neighbor by picking a random column and moving its queen
    to a different random row.
    """
    neighbor = list(state)
    col = random.randint(0, 7)
    current_row = state[col]

    # Pick a new row that is different from the current row
    new_row = random.randint(0, 7)
    while new_row == current_row:
        new_row = random.randint(0, 7)

    neighbor[col] = new_row
    return neighbor

def simulated_annealing_8_queens(initial_temp=100.0, cooling_rate=0.995, max_iterations=5000):
    # Initialize with a random board state (each queen in a random row 0-7)
    current_state = [random.randint(0, 7) for _ in range(8)]
    current_conflicts = calculate_conflicts(current_state)

    T = initial_temp

    print(f"Starting Simulated Annealing...")
    print(f"Initial State: {current_state} | Initial Conflicts: {current_conflicts}\n")

    for iteration in range(1, max_iterations + 1):
        # If solution is found, stop early
        if current_conflicts == 0:
            print(f"🎉 Solution found at iteration {iteration}!")
            return current_state

        # Cool down temperature
        T *= cooling_rate
        if T < 0.0001:
            # Temperature is practically zero; stop to avoid math underflow
            break

        # Get a random neighbor state
        neighbor = get_random_neighbor(current_state)
        neighbor_conflicts = calculate_conflicts(neighbor)

        # Energy difference (we want to minimize conflicts, so lower is better)
        delta_E = neighbor_conflicts - current_conflicts

        # If neighbor is better, always accept it.
        # If neighbor is worse, accept it with a probability determined by Temperature.
        if delta_E < 0:
            current_state = neighbor
            current_conflicts = neighbor_conflicts
        else:
            acceptance_probability = math.exp(-delta_E / T)
            if random.random() < acceptance_probability:
                current_state = neighbor
                current_conflicts = neighbor_conflicts

        # Log progress every 500 steps
        if iteration % 500 == 0:
            print(f"Iteration {iteration:4d} | Temp: {T:.4f} | Current Conflicts: {current_conflicts}")

    # Final check if max iterations or temperature minimum was reached
    if current_conflicts == 0:
        return current_state
    else:
        print("\nCould not find a perfect solution within the limits. Returning best effort state.")
        return current_state

def print_board(state):
    """Helper function to print the 8x8 chess grid layout."""
    for row in range(8):
        line = ""
        for col in range(8):
            if state[col] == row:
                line += " Q "
            else:
                line += " . "
        print(line)

# Run the algorithm
solution = simulated_annealing_8_queens()

print("\n--- FINAL RESULT ---")
print(f"Final State (Columns 0-7 Row Indices): {solution}")
print(f"Total Remaining Conflicts: {calculate_conflicts(solution)}")
print("\nVisual Board Layout:")
print_board(solution)
print('ankitha')
print('1wn24cs043')


Starting Simulated Annealing...
Initial State: [0, 5, 5, 2, 2, 2, 4, 4] | Initial Conflicts: 9

Iteration  500 | Temp: 8.1572 | Current Conflicts: 7
Iteration 1000 | Temp: 0.6654 | Current Conflicts: 2
🎉 Solution found at iteration 1034!

--- FINAL RESULT ---
Final State (Columns 0-7 Row Indices): [6, 3, 1, 7, 5, 0, 2, 4]
Total Remaining Conflicts: 0

Visual Board Layout:
 .  .  .  .  .  Q  .  . 
 .  .  Q  .  .  .  .  . 
 .  .  .  .  .  .  Q  . 
 .  Q  .  .  .  .  .  . 
 .  .  .  .  .  .  .  Q 
 .  .  .  .  Q  .  .  . 
 Q  .  .  .  .  .  .  . 
 .  .  .  Q  .  .  .  . 
ankitha
1wn24cs043


In [6]:
import pandas as pd
from itertools import product
print("1WN24CS043 ankitha ")
# Generate all possible truth value combinations
data = []

for P, Q, R in product([True, False], repeat=3):

    # Knowledge Base sentences
    S1 = (not Q) or P        # Q -> P
    S2 = (not P) or (not Q)  # P -> ~Q
    S3 = Q or R              # Q OR R

    # Knowledge Base
    KB = S1 and S2 and S3

    # Statements to check
    S4 = (not R) or P        # R -> P
    S5 = (not Q) or R        # Q -> R

    data.append([P, Q, R, S1, S2, S3, KB, S4, S5])

# Create truth table
columns = ['P', 'Q', 'R', 'Q -> P', 'P -> ~Q',
           'Q OR R', 'KB', 'R -> P', 'Q -> R']

df = pd.DataFrame(data, columns=columns)

# Convert Boolean values into T and F
df = df.map(lambda x: 'T' if x else 'F')

print("TRUTH TABLE")
display(df)

# Find models where KB is true
models = df[df['KB'] == 'T']

print("\nMODELS OF KB")
display(models[['P', 'Q', 'R']])

# Check entailments
print("\nENTAILMENT CHECKING")

print("ii) KB entails R:",
      all(models['R'] == 'T'))

print("iii) KB entails R -> P:",
      all(models['R -> P'] == 'T'))

print("iv) KB entails Q -> R:",
      all(models['Q -> R'] == 'T'))





1WN24CS043 ankitha 
TRUTH TABLE


,P,Q,R,Q -> P,P -> ~Q,Q OR R,KB,R -> P,Q -> R
0,T,T,T,T,F,T,F,T,T
1,T,T,F,T,F,T,F,T,F
2,T,F,T,T,T,T,T,T,T
3,T,F,F,T,T,F,F,T,T
4,F,T,T,F,T,T,F,F,T
5,F,T,F,F,T,T,F,T,F
6,F,F,T,T,T,T,T,F,T
7,F,F,F,T,T,F,F,T,T



MODELS OF KB


,P,Q,R
2,T,F,T
6,F,F,T



ENTAILMENT CHECKING
ii) KB entails R: True
iii) KB entails R -> P: False
iv) KB entails Q -> R: True
